In [4]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class TargetEncoder_:
    def transform(self, X):
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f"TE_{col}_{agg_func}"
                mapping = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(mapping)
                X_transformed[new_col_name] = X_transformed[new_col_name].fillna(
                    self.global_stats_[agg_func]
                )

        if getattr(self, "drop_original", False):
            X_transformed = X_transformed.drop(columns=self.cols_to_encode)

        return X_transformed


pipeline = joblib.load("stacking_ensemble_pipeline.pkl")
realmlp = pipeline["base_models"].get("RealMLP")
if realmlp is not None:
    realmlp.device = "cpu"
    realmlp.alg_interface_.device = "cpu"
train_reference = pd.read_csv("data/train-test.csv")
validation = pd.read_csv("data/validation.csv")
december = pd.read_csv("data/december-chart-inputs.csv")


def engineer_features(reference, inference):
    reference_features = reference.drop(columns=["load_id", "posted_rate"], errors="ignore")
    inference_features = inference.drop(columns=["load_id", "posted_rate"], errors="ignore").copy()

    coordinate_sources = {
        "pickup_lat": ("pickup", "pickup_lat"),
        "pickup_lon": ("pickup", "pickup_lon"),
        "delivery_lat": ("delivery", "delivery_lat"),
        "delivery_lon": ("delivery", "delivery_lon"),
    }
    for column in reference_features.columns:
        if column not in inference_features:
            if column in coordinate_sources:
                key_column, value_column = coordinate_sources[column]
                mapping = (
                    reference_features[[key_column, value_column]]
                    .dropna()
                    .drop_duplicates(key_column)
                    .set_index(key_column)[value_column]
                )
                inference_features[column] = inference_features[key_column].map(mapping)
            else:
                inference_features[column] = reference_features[column].mean()
    inference_features = inference_features.reindex(columns=reference_features.columns)

    reference_train, reference_test = train_test_split(
        reference_features, test_size=0.2, random_state=42
    )

    fill_values = reference_train[["weight", "market_index"]].mean()
    reference_train = reference_train.copy()
    reference_test = reference_test.copy()
    for frame in (reference_train, reference_test, inference_features):
        frame[["weight", "market_index"]] = frame[["weight", "market_index"]].fillna(fill_values)

    def add_features(frame):
        result = frame.copy()
        date = pd.to_datetime(result.pop("date"))
        result["Hour"] = date.dt.hour
        result["Month"] = date.dt.month
        result["Dayofweek"] = date.dt.dayofweek
        for column, period in {"Hour": 24, "Dayofweek": 7, "Month": 12}.items():
            result[f"_{column}_sin"] = np.sin(2 * np.pi * result[column] / period).astype("float32")
            result[f"_{column}_cos"] = np.cos(2 * np.pi * result[column] / period).astype("float32")
        result = result.drop(columns=["Hour", "Dayofweek", "Month"])
        for column in ["weight", "distance"]:
            result[f"{column}_sq"] = result[column] ** 2
            result[f"{column}_2"] = result[column].copy()
        result["feature_formula"] = (
            1.8696 * result["distance"]
            + 0.0073 * result["weight"]
            + 329.5113 * result["market_index"]
            + 43.6723 * result["quote_signal"]
            + 177.9682 * (result["equipment"] == "Flatbed").astype(float)
            + 284.7715 * (result["equipment"] == "Reefer").astype(float)
            - 447.4607
        )
        result["equipment_delivery_hub"] = (
            result["equipment"].astype(str) + "_" + result["delivery"].astype(str)
        )
        return result

    reference_train = add_features(reference_train)
    reference_test = add_features(reference_test)
    inference_features = add_features(inference_features)

    combined = pd.concat(
        [
            reference_train["equipment_delivery_hub"],
            reference_test["equipment_delivery_hub"],
            inference_features["equipment_delivery_hub"],
        ],
        ignore_index=True,
    )
    encoded_values, _ = combined.factorize()
    inference_features["equipment_delivery_hub"] = encoded_values[
        len(reference_train) + len(reference_test):
    ].astype("int16")
    return inference_features


def predict(frame):
    features = engineer_features(train_reference, frame)
    encoded = pipeline["target_encoder"].transform(features)
    prediction_matrix = pd.DataFrame(
        {
            name: model.predict(encoded)
            for name, model in pipeline["base_models"].items()
        }
    )
    return pipeline["meta_model"].predict(prediction_matrix)


validation_predictions = predict(validation)
validation_output = pd.DataFrame(
    {
        "load_id": validation["load_id"],
        "predicted_rate": np.maximum(validation_predictions, 0.01),
    }
)
validation_output.to_csv("validation_predictions.csv", index=False)


december_predictions = predict(december)
december_output = december.copy()
december_output["predicted_rate"] = np.maximum(december_predictions, 0.01)
december_output.to_csv("data/december-chart-inputs.csv", index=False)

print(f"Saved {len(validation_output):,} validation predictions to validation_predictions.csv")
print(f"Saved {len(december_output):,} December predictions to data/december-chart-inputs.csv")
print(validation_output.head())

g:\Interview Prep\Projeenv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
g:\Interview Prep\Projeenv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
g:\Interview Prep\Projeenv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.6.1 when using version 1.9.0. This might 

Saved 12,000 validation predictions to validation_predictions.csv
Saved 31 December predictions to data/december-chart-inputs.csv
     load_id  predicted_rate
0  TE-000001      833.107179
1  TE-000002     4834.767854
2  TE-000003     5472.340660
3  TE-000004     4001.806937
4  TE-000005     1808.305970


g:\Interview Prep\Projeenv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
